<a href="https://colab.research.google.com/github/LUNALEE002/-/blob/main/MicroDuck_Colab_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐥 MicroDuck 강화학습(RL) Google Colab 훈련 노트북

Pollen Robotics의 MicroDuck 로봇을 MuJoCo Warp 기반으로 GPU 병렬 강화학습(PPO) 훈련을 수행하고, 완성된 모델을 ONNX 형식으로 내보냅니다.

> ⚠️ **중요**: 상단 메뉴의 **[런타임] → [런타임 유형 변경]**에서 하드웨어 가속기를 **T4 GPU** (또는 A100/L4)로 설정했는지 확인하세요.

### 1. GPU 할당 확인

In [8]:
!nvidia-smi

Sun Sep  6 01:09:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. uv 설치 및 저장소 클론

In [9]:
# uv 초고속 패키지 관리자 설치
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

# Microduck 저장소 클론
!git clone https://github.com/pollen-robotics/microduck_rl.git
!git clone https://github.com/pollen-robotics/microduck.git

downloading uv 0.12.10 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Cloning into 'microduck_rl'...
remote: Enumerating objects: 14485, done.
remote: Counting objects: 100% (2485/2485), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 14485 (delta 2398), reused 2276 (delta 2276), pack-reused 12000 (from 3)
Receiving objects: 100% (14485/14485), 52.00 MiB | 16.63 MiB/s, done.
Resolving deltas: 100% (9300/9300), done.
Cloning into 'microduck'...
remote: Enumerating objects: 7262, done.
remote: Counting objects: 100% (1802/1802), done.
remote: Compressing objects: 100% (488/488), done.
remote: Total 7262 (delta 1508), reused 1326 (delta 1313), pack-reused 5460 (from 2)
Receiving objects: 100% (7262/7262), 26.26 MiB | 10.21 MiB/s, done.
Resolving deltas: 100% (4467/4467), done.


### 3. 의존성 설치 및 가상환경 구성

In [10]:
%cd /content/microduck_rl
!uv sync

/content/microduck_rl
Using CPython 3.12.14
Creating virtual environment at: .venv
Resolved 154 packages in 1ms
Prepared 142 packages in 1m 39s
Installed 142 packages in 1.23s
 + absl-py==2.3.1
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.12.1
 + asttokens==3.0.1
 + bcrypt==5.0.0
 + better-actuator-models==1.0.1 (from git+https://github.com/Rhoban/bam.git@62bd8ce12154340be97e06f7f41a0ca8f116d967)
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + cloudpickle==3.1.2
 + colorama==0.4.6
 + contourpy==1.3.3
 + cryptography==46.0.3
 + cycler==0.12.1
 + decorator==5.2.1
 + deprecated==1.3.1
 + docstring-parser==0.17.0
 + etils==1.13.0
 + executing==2.2.1
 + fabric==3.2.2
 + filelock==3.20.3
 + flatbuffers==25.12.19
 + fonttools==4.63.0
 + fsspec==2026.1.0
 + gitdb==4.0.12
 + gitpython==3.1.46
 + glfw==2.10.0
 + grpcio==1.76.0
 + h11==0.16.0
 + hf-xet==1.5.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.15.0
 + idna==3.11
 + imagei

### 4. 강화학습 훈련 (Train)

- 기본 4,096개 병렬 환경에서 보행 정책을 학습합니다.
- 메모리(VRAM) 이슈가 발생할 경우 `--env.scene.num-envs 2048`로 줄여서 실행하세요.

In [11]:
# 2,000회까지 훈련 (Resume 없이 새로 시작)
%cd /content/microduck_rl
!WANDB_MODE=disabled WANDB_DISABLED=true uv run python src/mjlab_microduck/train_cli.py Mjlab-Velocity-Flat-MicroDuck --env.scene.num-envs 4096 --agent.max-iterations 2000

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
                   Curriculum/com_range: 0.0150
              Curriculum/head_com_range: 0.0100
       Curriculum/head_pose_bias_weight: 3.0000
             Metrics/twist/error_vel_xy: 0.4191
            Metrics/twist/error_vel_yaw: 1.0806
           Episode_Termination/time_out: 3.1667
          Episode_Termination/fell_over: 0.5833
Episode_Termination/out_of_terrain_bounds: 0.0000
          Episode_Termination/nan_state: 0.0000
          Metrics/angular_momentum_mean: 0.0035
                  Metrics/air_time_mean: 0.1212
               Metrics/peak_height_mean: 0.0134
             Metrics/slip_velocity_mean: 0.0225
--------------------------------------------------------------------------------
                         Iteration time: 4.75s
                           Time elapsed: 02:32:53
                                    ETA: 00:07:32

################################################################################
                          Le

### 5. 최신 체크포인트를 ONNX 모델로 Export

In [12]:
import glob
import os
import re

# 실제 로그가 저장된 경로
base_log_path = "/content/microduck_rl/logs/rsl_rl/velocity/*"
log_dirs = sorted(glob.glob(base_log_path))

if log_dirs:
    latest_run_path = log_dirs[-1]
    latest_run_name = os.path.basename(latest_run_path)
    print(f"최신 학습 디렉토리 감지: {latest_run_name}")

    # 해당 디렉토리 내의 체크포인트(.pt) 파일 찾기 및 숫자 기준 정렬
    checkpoint_files = glob.glob(os.path.join(latest_run_path, "model_*.pt"))

    if not checkpoint_files:
        print("❌ 체크포인트(.pt) 파일을 찾을 수 없습니다.")
    else:
        # 파일명에서 숫자를 추출하여 숫자 크기 순으로 정렬
        checkpoint_files.sort(key=lambda x: int(re.findall(r'\d+', os.path.basename(x))[0]))
        latest_checkpoint = checkpoint_files[-1]
        print(f"사용할 최신 체크포인트: {os.path.basename(latest_checkpoint)}")

        %cd /content/microduck_rl
        # ONNX 변환 실행
        !uv run scripts/export.py Mjlab-Velocity-Flat-MicroDuck --checkpoint-file {latest_checkpoint} --onnx-file custom_walking.onnx

        if os.path.exists("custom_walking.onnx"):
            print("✅ custom_walking.onnx 파일 생성 완료!")
        else:
            print("❌ ONNX 파일 생성에 실패했습니다.")
else:
    print("❌ 학습 로그 디렉토리를 찾을 수 없습니다.")

최신 학습 디렉토리 감지: 2026-09-06_01-18-30_velocity
사용할 최신 체크포인트: model_1999.pt
/content/microduck_rl
[mdp] Patches 1-2 active: NaN-safe reward/advantage
[mdp] Patch 4 active: ONNX export filters passive_* joints
[INFO]: Loading checkpoint: model_1999.pt
/content/microduck_rl/.venv/lib/python3.12/site-packages/mjlab/entity/entity.py:149: UserWarning: Actuator config matched 14 joint(s) but the same expressions also match 7 site(s): imu, left_foot, head_camera, tof, head_imu, mouth_tip, right_foot. Add a separate config with transmission_type=TransmissionType.SITE if those should be actuated too.
  self._add_actuators()
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.0
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be ch

### 6. 생성된 ONNX 모델 로컬 PC로 다운로드

In [13]:
from google.colab import files
import os

# 파일의 절대 경로 지정
onnx_file_path = "/content/microduck_rl/custom_walking.onnx"

if os.path.exists(onnx_file_path):
    files.download(onnx_file_path)
    print(f"{onnx_file_path} 다운로드를 시작합니다.")
else:
    print(f"❌ 파일을 찾을 수 없습니다: {onnx_file_path}")
    # 혹시 모르니 현재 디렉토리 파일 목록 확인
    !ls -l /content/microduck_rl/*.onnx

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/microduck_rl/custom_walking.onnx 다운로드를 시작합니다.
